# Softmax Classifier for CIFAR-10 Image Classification

## What this notebook demonstrates

- CIFAR-10 image classification with a linear softmax classifier
- cross-entropy loss for multiclass classification
- numerical and analytic gradients
- vectorized loss and gradient computation
- stochastic gradient descent and model evaluation

## Academic context

This notebook is a cleaned portfolio version of MSc coursework and implementation practice. It is included to demonstrate foundational computer vision and deep learning skills.

## Local helper package

This notebook uses the included project-local `engine/` package for coursework helper code such as data loading, classifiers, layers, optimization, and visualization utilities.


In [ ]:
# Project-local setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "engine").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the project root containing the engine/ package.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")


### Set up code


In [ ]:
# Run some setup code for this notebook.
import random
import numpy as np
import matplotlib.pyplot as plt

# Display matplotlib figures inline.
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Reload local helper modules during iterative notebook work.
%load_ext autoreload
%autoreload 2


### CIFAR-10 Data Loading and Pre-processing


⚠️ **NOTICE**: In case you have already stored CIFAR-10 dataset in ``.npy`` files (as done in ``svm.ipynb``), you can **SKIP** this code snippet.


In [ ]:
from engine.data_utils import load_CIFAR10

# Load the raw CIFAR-10 data.
cifar10_dir = str(DATA_DIR / 'cifar-10-batches-py')

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


### Load .npy files


Load dataset from ``.npy`` files! This way is really faster!


In [ ]:
# Specify data directory, so that arrays dir is
# automatically retrieved and arrays can be loaded from it
data_dir = str(DATA_DIR)
arrays_dir = os.path.join(data_dir, 'arrays')

# Load arrays if needed. Uncomment to use!
X_train = np.load(os.path.join(arrays_dir, 'full_X_train.npy'))
y_train = np.load(os.path.join(arrays_dir, 'full_y_train.npy'))
X_test = np.load(os.path.join(arrays_dir, 'full_X_test.npy'))
y_test = np.load(os.path.join(arrays_dir, 'full_y_test.npy'))

# Num of samples for training and test
num_training = X_train.shape[0]
num_test = X_test.shape[0]


### Split data


In [ ]:
# Split the data into train, val, and test sets. In addition we will
# create a small development set as a subset of the training data.
# We can use this for development so our code runs faster.
num_training = 45000
num_validation = 5000
num_test = 10000
num_dev = 500

# Our validation set will be num_validation points from the original
# training set.
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Our training set will be the first num_train points from the original
# training set.
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# We will also make a development set, which is a small subset of
# the training set.
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# We use the first num_test points of the original test set as our
# test set.
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)


### Pre-process data


In [ ]:
# Preprocessing: reshape the image data into rows
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# As a sanity check, print out the shapes of the data
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)


In [ ]:
# Preprocessing: subtract the mean image
# first: compute the image mean based on the training data
mean_image = np.mean(X_train, axis=0)

# Visualize some of its elements and the image itself
print('Some of the elements of the mean image:')
print(mean_image[:10]) # print a few of the elements
print('The mean image of our train set:')
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # visualize the mean image
plt.show()

# second: subtract the mean image from train and test data
mean_image = mean_image.astype('uint8')
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# third: append the bias dimension of ones (i.e. bias trick) so that our SVM
# only has to worry about optimizing a single weight matrix W.
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])


Let's see our arrays' new shapes after bias trick.


In [ ]:
print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)


### Softmax Classifier


#### Forward and Backward (naive)


This section uses the implementation in `engine/classifiers/softmax.py`.

**Implementation note.**

This section validates the ``softmax_loss_naive()`` function.

This begins with the softmax loss before checking the gradient.


In [ ]:
# Compute the naive softmax loss with nested loops.

from engine.classifiers.softmax import softmax_loss_naive
import time

# Generate a random softmax weight matrix and use it to compute the loss.
W = np.random.randn(3073, 10) * 0.0001
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# As a rough sanity check, our loss should be something close to -log(0.1).
print('loss: %f' % loss)
print('sanity check: %f' % (-np.log(0.1)))


**Concept check.**

Why do we expect our loss to be close to -log(0.1)? Explain briefly.

**Answer.** A loss close to -log(0.1) reflects a model that is guessing randomly, as it assigns equal confidence to all classes without learning meaningful patterns in the data.


**Implementation note.**

This section checks the naive softmax gradient computed with explicit loops.


#### Gradient check


In [ ]:
# Compute the naive softmax loss and gradient
# version of the gradient that uses nested loops.
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# As we did for the SVM, use numeric gradient checking as a debugging tool.
# The numeric gradient should be close to the analytic gradient.
from engine.gradient_check import grad_check_sparse
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad, 10)

# similar to SVM case, do another gradient check with regularization
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad, 10)


#### Forward and Backward (vectorized)


**Implementation note.**

This section validates the vectorized softmax loss and gradient in `engine/classifiers/softmax.py`.


In [ ]:
# Compare the naive softmax implementation with the vectorized implementation.
# The two versions should compute the same results, but the vectorized version should be
# much faster.
tic = time.time()
loss_naive, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from engine.classifiers.softmax import softmax_loss_vectorized
tic = time.time()
loss_vectorized, grad_vectorized = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# As we did for the SVM, we use the Frobenius norm to compare the two versions
# of the gradient.
grad_difference = np.linalg.norm(grad_naive - grad_vectorized, ord='fro')
print('Loss difference: %f' % np.abs(loss_naive - loss_vectorized))
print('Gradient difference: %f' % grad_difference)


#### Hyperparameter search


**Implementation note.**

Write below code that chooses the best hyperparameters (regularization strength and
learning rate) by tuning on the validation set (grid search). Your goal is to perform a real-world training in order to save the softmax model that performs the best on validation set.


In [ ]:
# Use the validation set to tune hyperparameters (regularization strength and
# learning rate). Expected: experiment with different ranges for the learning
# rates and regularization strengths; careful tuning can
# get a classification accuracy of over 0.35 on the validation set.

from engine.classifiers import Softmax
results = {}
best_val = -1
best_softmax = None

# Use the validation set to set the learning rate and regularization strength. #
# This should be identical to the validation that you did for the SVM; save    #
# the best trained softmax classifer in best_softmax.                          #

# Provided as a reference. You may or may not want to change these hyperparameters
learning_rates = [1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
regularization_strengths = [1e-6, 5e-6, 1e-5, 5e-5, 1e-4, 5e-4, 1e-3, 5e-3, 1e-2]

grid_search = [(lr,rg) for lr in learning_rates for rg in regularization_strengths]

for lr, rg in grid_search:
    softmax = Softmax()
    train_loss = softmax.train(X_train, y_train, learning_rate=lr, reg=rg,
                      num_iters=1500, verbose=False)

    y_train_pred = softmax.predict(X_train)
    train_accuracy = np.mean(y_train_pred == y_train)
    y_val_pred = softmax.predict(X_val)

    val_accuracy = np.mean(y_val_pred == y_val)
    results[(lr,rg)] = (train_accuracy, val_accuracy)
    if best_val < val_accuracy:
        best_val = val_accuracy
        best_softmax = softmax

# Print out results.
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during validation: %f' % best_val)


#### Predict on test set (unseen images)


Let's see how our model performs on unseen images - the test set.


In [ ]:
# Evaluate the best softmax on test set
y_test_pred = best_softmax.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('softmax on raw pixels final test set accuracy: %f' % (test_accuracy, ))


**Concept check.** *True or False*

Suppose the overall training loss is defined as the sum of the per-datapoint loss over all training examples. It is possible to add a new datapoint to a training set that would leave the SVM loss unchanged, but this is not the case with the Softmax classifier loss.

**Answer.** True


**Explanation.** If a new datapoint is added to the training set such that its scores satisfy the margin condition, the hinge loss for that data point is zero. In this case, the addition of this datapoint does not contribute to the overall loss because it does not violate the margin. The softmax loss does not have a concept of a "margin." It always assigns a loss value to a datapoint based on the softmax probabilities, even if the model classifies the point correctly with high confidence. This means that adding any new datapoint to the training set always changes the total softmax loss.


#### Visualization of learned weights for each class


In [ ]:
# Visualize the learned weights for each class
w = best_softmax.W[:-1,:] # strip out the bias
w = w.reshape(32, 32, 3, 10)

w_min, w_max = np.min(w), np.max(w)

classes = ['Plane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # Rescale the weights to be between 0 and 255
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])
